# RAGScholar — Notebook 02  
## Content Readiness & Challenge-Set Certification

**Purpose:** complete the dataset-checking phase before any retrieval-model, Math-RAG, Visual-RAG, embedding, reranking, or full RAG implementation research begins.

Notebook 01 certified the physical corpus. This notebook answers a different question:

> **Does the certified corpus contain machine-recoverable text, mathematics, visuals, tables, layouts, and source locators in a form suitable for the next RAG research notebook?**

### What this notebook does

- Re-validates the frozen 53-document corpus snapshot.
- Performs a full page-level content census across the entire corpus.
- Profiles text, raster images, vector graphics, mathematical signals, tables, and layout complexity.
- Detects image-only/scanned pages, multi-column pages, structural noise, and extraction anomalies.
- Audits visual assets without deciding the final Visual-RAG model.
- Audits mathematical content without deciding the final Math-RAG method.
- Builds a high-quality challenge/sentinel page set for later parser, text, math, table, and visual research.
- Certifies whether the dataset is ready to move into the main RAG research/implementation stage.

### Deliberately out of scope

This notebook **does not** choose embedding models, visual embedding models, Math-RAG models, parsers, rerankers, or RAGFlow configuration. Those are research/implementation decisions for the next notebook.

## 0. Setup & Audit Configuration

All paths are project-relative. The notebook expects the final artifacts created by Notebook 01 under `metadata/`.

In [2]:
from pathlib import Path
from IPython.display import display
from collections import Counter
import pandas as pd
import numpy as np
import fitz
import hashlib
import json
import re
import sys
import platform
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 140)

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs): return x

TARGET = {"Textbooks": 6, "Survey": 23, "Landmark": 24}
TARGET_TOTAL = sum(TARGET.values())

PAPER_CHALLENGE_PAGES = 4
TEXTBOOK_CHALLENGE_PAGES = 6
GLOBAL_EXTRA_CHALLENGE_PAGES = 20
TABLE_VALIDATION_LIMIT = 600
EXPORT_CHALLENGE_PREVIEWS = False
PREVIEW_DPI = 110

MATH_SYMBOLS = set("αβγδεζηθικλμνξοπρστυφχψωΓΔΘΛΞΠΣΦΨΩ∑∏∫√∇∂≤≥≈≠±×÷∞→←↔⊙∈∉⊂⊆∪∩⊗⊕")
MATH_FONT_HINTS = ("math", "symbol", "cmsy", "cmmi", "cmr", "stix", "latinmodern", "msam", "msbm", "euler")
FIGURE_RE = re.compile(r"(?im)^\s*(?:fig(?:ure)?\.?\s*\d+[a-z]?|diagram\s*\d+[a-z]?)\s*[:.\-]?")
TABLE_RE = re.compile(r"(?im)^\s*table\s+\d+[a-z]?\s*[:.\-]?")
ALGORITHM_RE = re.compile(r"(?im)^\s*algorithm\s+\d+[a-z]?\s*[:.\-]?")
REFERENCE_RE = re.compile(r"(?im)^\s*(references|bibliography)\s*$")
TOC_RE = re.compile(r"(?im)^\s*(table\s+of\s+contents|contents)\s*$")
INDEX_RE = re.compile(r"(?im)^\s*index\s*$")

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in candidates if (p/"metadata"/"corpus_manifest.csv").exists()), None)
assert PROJECT_ROOT is not None, "Project root not found. Expected metadata/corpus_manifest.csv."

PROJECT_ROOT = PROJECT_ROOT.resolve()
METADATA_DIR = PROJECT_ROOT/"metadata"
OUT_DIR = METADATA_DIR/"notebook02"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR :", OUT_DIR)
print("PyMuPDF    :", fitz.VersionBind)
print("Python     :", platform.python_version())

PROJECT_ROOT: C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt
OUTPUT_DIR : C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\metadata\notebook02
PyMuPDF    : 1.28.0
Python     : 3.10.20


## 1. Certified Corpus & Snapshot Gate

Notebook 02 must start from exactly the corpus that Notebook 01 certified.  
This cell checks the manifest, category counts, final certification report, physical file hashes, and the frozen corpus snapshot.

In [3]:
manifest = pd.read_csv(METADATA_DIR/"corpus_manifest.csv")

with open(METADATA_DIR/"final_certification_report.json", encoding="utf-8") as f:
    certification_01 = json.load(f)

with open(METADATA_DIR/"corpus_snapshot.sha256", encoding="utf-8") as f:
    saved_snapshot = f.read().strip()

def resolve_pdf_path(row):
    rel = PROJECT_ROOT/str(row["relative_path"])
    abs_path = Path(str(row["absolute_path"]))
    return rel.resolve() if rel.exists() else abs_path

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(1024*1024): h.update(chunk)
    return h.hexdigest()

manifest["resolved_path"] = manifest.apply(resolve_pdf_path, axis=1)
manifest["file_exists_now"] = manifest["resolved_path"].apply(Path.exists)
manifest["sha256_now"] = manifest["resolved_path"].apply(lambda p: sha256_file(p) if Path(p).exists() else "")
manifest["hash_match_now"] = manifest["sha256_now"].str.lower() == manifest["sha256"].astype(str).str.lower()

snapshot_payload = "\n".join(f"{r['canonical_work_id']}|{r['sha256_now']}" for _, r in manifest.sort_values("canonical_work_id").iterrows())
snapshot_now = hashlib.sha256(snapshot_payload.encode("utf-8")).hexdigest()

category_counts = manifest["category"].value_counts().to_dict()
gate_01 = {
    "Notebook 01 certified": certification_01.get("certification", "").startswith("CERTIFIED"),
    "Notebook 01 ready": certification_01.get("ready_for_notebook_02") is True,
    "53 manifest documents": len(manifest) == TARGET_TOTAL,
    "Category counts exact": all(category_counts.get(k, 0) == v for k, v in TARGET.items()),
    "All physical files exist": manifest["file_exists_now"].all(),
    "All physical hashes unchanged": manifest["hash_match_now"].all(),
    "Corpus snapshot unchanged": snapshot_now == saved_snapshot
}

gate_01_df = pd.DataFrame([{"check": k, "status": "PASS" if v else "FAIL"} for k, v in gate_01.items()])
display(gate_01_df)

assert all(gate_01.values()), "Notebook 01 corpus state changed. Resolve the failed gate before continuing."
print("CERTIFIED CORPUS SNAPSHOT:", snapshot_now)
print("STATUS: PASS ✅")

,check,status
0,Notebook 01 certified,PASS
1,Notebook 01 ready,PASS
2,53 manifest documents,PASS
3,Category counts exact,PASS
4,All physical files exist,PASS
5,All physical hashes unchanged,PASS
6,Corpus snapshot unchanged,PASS


CERTIFIED CORPUS SNAPSHOT: 09e5e430bcdcc874ebe4291fca74594e4fa91264723a0f54571bac8fdfc80194
STATUS: PASS ✅


## 2. Full Page-Level Content Census

Every physical page is scanned once with a lightweight baseline extractor.  
The goal is **content inventory**, not final parsing fidelity.

For each page we record:

- text/word/block counts;
- page geometry and printed page label;
- raster-image coverage;
- vector-drawing density;
- mathematical-symbol and math-font signals;
- figure/table/algorithm caption signals;
- numeric density;
- likely multi-column structure;
- reference/TOC/index noise signals;
- Unicode replacement-character anomalies.

This produces the master page-level audit table used by the rest of Notebook 02.

In [4]:
def safe_label(page):
    try: return page.get_label() or ""
    except Exception: return ""

def image_info_safe(page):
    try:
        infos = page.get_image_info(xrefs=True)
        return infos if infos else []
    except Exception:
        infos = []
        try:
            for item in page.get_images(full=True):
                xref = item[0]
                rects = page.get_image_rects(xref)
                for rect in rects:
                    infos.append({"xref": xref, "bbox": tuple(rect), "width": item[2], "height": item[3], "bpc": item[4], "colorspace": item[5], "digest": ""})
        except Exception:
            pass
        return infos

def two_column_hint(words, width):
    if not words or width <= 0: return False, 0.0
    centers = np.array([(w[0]+w[2])/2 for w in words], dtype=float)
    left = int((centers < width*0.45).sum())
    right = int((centers > width*0.55).sum())
    middle = len(centers)-left-right
    middle_ratio = middle/max(len(centers), 1)
    return bool(left >= 20 and right >= 20 and middle_ratio <= 0.18), round(middle_ratio, 3)

def equation_like_count(text):
    lines = [re.sub(r"\s+", " ", x).strip() for x in text.splitlines() if x.strip()]
    count = 0
    for line in lines:
        if len(line) > 280: continue
        math_chars = sum(ch in MATH_SYMBOLS for ch in line)
        operators = len(re.findall(r"(?:=|≈|≤|≥|≠|±|×|÷|→|←|\^|_[a-zA-Z0-9{])", line))
        variable_pattern = bool(re.search(r"\b[A-Za-z](?:_[A-Za-z0-9]+|\^\{?[A-Za-z0-9+\-]+\}?)", line))
        if math_chars >= 1 or operators >= 2 or (operators >= 1 and variable_pattern): count += 1
    return count

page_rows, visual_rows, scan_errors = [], [], []

for _, docrow in tqdm(manifest.iterrows(), total=len(manifest), desc="Scanning documents"):
    path = Path(docrow["resolved_path"])
    try:
        with fitz.open(path) as doc:
            for page_index in range(doc.page_count):
                try:
                    page = doc.load_page(page_index)
                    rect = page.rect
                    page_area = max(rect.width*rect.height, 1)
                    text = page.get_text("text") or ""
                    words = page.get_text("words") or []
                    raw = page.get_text("dict") or {}
                    blocks = raw.get("blocks", [])
                    spans = [s for b in blocks for line in b.get("lines", []) for s in line.get("spans", [])]
                    images = image_info_safe(page)

                    image_area = 0.0
                    for img_no, info in enumerate(images, 1):
                        bbox = info.get("bbox", (0,0,0,0))
                        try:
                            x0, y0, x1, y1 = map(float, bbox)
                            area_ratio = max(0.0, (x1-x0)*(y1-y0)/page_area)
                        except Exception:
                            x0=y0=x1=y1=area_ratio=0.0
                        digest = info.get("digest", "")
                        if isinstance(digest, bytes): digest = digest.hex()
                        image_area += area_ratio
                        visual_rows.append({
                            "inventory_id": docrow["inventory_id"], "canonical_work_id": docrow["canonical_work_id"], "filename": docrow["filename"],
                            "category": docrow["category"], "page_index": page_index, "pdf_page_number": page_index+1, "printed_page_label": safe_label(page),
                            "image_no": img_no, "xref": info.get("xref", 0), "width_px": info.get("width", 0), "height_px": info.get("height", 0),
                            "bpc": info.get("bpc", 0), "colorspace": info.get("colorspace", 0), "bbox_x0": x0, "bbox_y0": y0, "bbox_x1": x1, "bbox_y1": y1,
                            "area_ratio": round(area_ratio, 5), "digest": str(digest)
                        })

                    try: drawing_count = len(page.get_drawings())
                    except Exception: drawing_count = 0

                    math_symbol_count = sum(ch in MATH_SYMBOLS for ch in text)
                    math_font_spans = sum(any(h in str(s.get("font", "")).lower() for h in MATH_FONT_HINTS) for s in spans)
                    eq_like = equation_like_count(text)
                    figure_captions = len(FIGURE_RE.findall(text))
                    table_captions = len(TABLE_RE.findall(text))
                    algorithm_captions = len(ALGORITHM_RE.findall(text))
                    replacement_chars = text.count("\ufffd") + text.count("�") + text.count("□")
                    numeric_tokens = sum(bool(re.fullmatch(r"[-+]?\d+(?:[.,]\d+)*(?:%|[eE][-+]?\d+)?", str(w[4]).strip())) for w in words)
                    numeric_ratio = numeric_tokens/max(len(words), 1)
                    is_two_col, middle_ratio = two_column_hint(words, rect.width)

                    page_rows.append({
                        "inventory_id": docrow["inventory_id"], "canonical_work_id": docrow["canonical_work_id"], "filename": docrow["filename"],
                        "canonical_title": docrow["canonical_title"], "category": docrow["category"], "source_type": docrow["source_type"],
                        "primary_topic": docrow["primary_topic"], "page_index": page_index, "pdf_page_number": page_index+1,
                        "printed_page_label": safe_label(page), "width": round(rect.width, 2), "height": round(rect.height, 2), "rotation": page.rotation,
                        "char_count": len(text.strip()), "word_count": len(words), "block_count": len(blocks), "span_count": len(spans),
                        "raster_image_count": len(images), "raster_area_ratio": round(min(image_area, 1.0), 4), "vector_drawing_count": drawing_count,
                        "math_symbol_count": math_symbol_count, "math_font_spans": math_font_spans, "equation_like_lines": eq_like,
                        "figure_caption_count": figure_captions, "table_caption_count": table_captions, "algorithm_caption_count": algorithm_captions,
                        "numeric_token_ratio": round(numeric_ratio, 4), "two_column_hint": is_two_col, "column_middle_ratio": middle_ratio,
                        "reference_heading": bool(REFERENCE_RE.search(text)), "toc_heading": bool(TOC_RE.search(text)), "index_heading": bool(INDEX_RE.search(text)),
                        "replacement_chars": replacement_chars, "scan_error": ""
                    })
                except Exception as e:
                    scan_errors.append({"inventory_id": docrow["inventory_id"], "filename": docrow["filename"], "page_index": page_index, "error": str(e)})
    except Exception as e:
        scan_errors.append({"inventory_id": docrow["inventory_id"], "filename": docrow["filename"], "page_index": None, "error": str(e)})

pages_df = pd.DataFrame(page_rows)
visual_assets_df = pd.DataFrame(visual_rows)
scan_errors_df = pd.DataFrame(scan_errors, columns=["inventory_id", "filename", "page_index", "error"])

expected_pages = int(manifest["page_count"].sum())

print("Expected physical pages :", expected_pages)
print("Profiled physical pages :", len(pages_df))
print("Page scan errors        :", len(scan_errors_df))
print("Raster asset occurrences:", len(visual_assets_df))

assert len(pages_df) == expected_pages and scan_errors_df.empty, "Full page census is incomplete."
print("STATUS: PASS ✅")

Scanning documents:   0%|          | 0/53 [00:00<?, ?it/s]

Expected physical pages : 6794
Profiled physical pages : 6794
Page scan errors        : 0
Raster asset occurrences: 9768
STATUS: PASS ✅


## 3. Page Modality & Complexity Profiling

Each page receives **heuristic audit scores** for text, visual, math, table, and layout complexity.

These labels are **not parser ground truth**. They exist only to understand the dataset and select difficult/representative pages for the next research notebook.

In [5]:
p = pages_df

p["text_score"] = np.clip(np.log1p(p["char_count"])/np.log(5000), 0, 1)
p["visual_score"] = np.clip(p["raster_area_ratio"]*1.15 + np.minimum(p["vector_drawing_count"]/120, 0.55) + np.minimum(p["figure_caption_count"]*0.20, 0.35), 0, 1)
p["math_score"] = np.clip(np.minimum(p["math_symbol_count"]/18, 0.35) + np.minimum(p["equation_like_lines"]/8, 0.40) + np.minimum(p["math_font_spans"]/18, 0.35), 0, 1)
p["table_score"] = np.clip(np.minimum(p["table_caption_count"]*0.45, 0.55) + np.minimum(p["vector_drawing_count"]/150, 0.25) + np.minimum(p["numeric_token_ratio"]/0.25, 0.35), 0, 1)
p["layout_score"] = np.clip(p["two_column_hint"].astype(float)*0.55 + np.minimum(p["block_count"]/100, 0.20) + np.minimum(p["raster_image_count"]/6, 0.15) + np.minimum(p["vector_drawing_count"]/180, 0.25), 0, 1)

def classify_page(r):
    active = sum([r["visual_score"] >= 0.40, r["math_score"] >= 0.40, r["table_score"] >= 0.45])
    if r["char_count"] < 30 and r["visual_score"] < 0.20: return "BLANK_OR_MINIMAL"
    if r["char_count"] < 120 and r["visual_score"] >= 0.45: return "VISUAL_DOMINANT"
    if active >= 2: return "MIXED_COMPLEX"
    if r["table_score"] >= 0.55: return "TABLE_RICH"
    if r["math_score"] >= 0.45: return "MATH_RICH"
    if r["visual_score"] >= 0.45: return "VISUAL_RICH"
    if r["text_score"] >= 0.45: return "TEXT_RICH"
    return "MIXED_LIGHT"

pages_df["page_modality"] = pages_df.apply(classify_page, axis=1)
pages_df["difficulty_score"] = np.clip(0.28*pages_df["layout_score"] + 0.24*pages_df["math_score"] + 0.22*pages_df["visual_score"] + 0.18*pages_df["table_score"] + 0.08*(pages_df["replacement_chars"] > 0).astype(float), 0, 1)

modality_summary = pages_df["page_modality"].value_counts().rename_axis("page_modality").reset_index(name="pages")
display(modality_summary)
display(pages_df.nlargest(20, "difficulty_score")[["inventory_id", "filename", "pdf_page_number", "printed_page_label", "page_modality", "text_score", "visual_score", "math_score", "table_score", "layout_score", "difficulty_score"]])

,page_modality,pages
0,MATH_RICH,2421
1,TEXT_RICH,1997
2,MIXED_COMPLEX,1635
3,VISUAL_RICH,574
4,BLANK_OR_MINIMAL,77
5,TABLE_RICH,57
6,VISUAL_DOMINANT,22
7,MIXED_LIGHT,11


,inventory_id,filename,pdf_page_number,printed_page_label,page_modality,text_score,visual_score,math_score,table_score,layout_score,difficulty_score
541,S007,survey_16.pdf,24,,MIXED_COMPLEX,0.962986,0.924840,1.000000,1.0000,1.00,0.903465
97,L002,Landmark_10.pdf,5,,MIXED_COMPLEX,0.970557,0.900000,1.000000,1.0000,1.00,0.898000
218,L022,Landmark_7.pdf,6,,MIXED_COMPLEX,0.961919,0.875000,1.000000,1.0000,1.00,0.892500
464,S010,survey_19.pdf,8,,MIXED_COMPLEX,1.000000,0.980805,1.000000,0.8636,1.00,0.891225
230,L022,Landmark_7.pdf,18,,MIXED_COMPLEX,0.924302,0.977970,0.861111,1.0000,1.00,0.881820
1272,S001,survey_1.pdf,23,,MIXED_COMPLEX,0.974562,0.799795,1.000000,1.0000,1.00,0.875955
3334,T001,book1.pdf,463,433,MIXED_COMPLEX,0.903769,0.900000,1.000000,0.9244,0.94,0.867592
98,L002,Landmark_10.pdf,6,,MIXED_COMPLEX,0.989056,0.750000,1.000000,1.0000,1.00,0.865000
465,S010,survey_19.pdf,9,,MIXED_COMPLEX,1.000000,0.750000,1.000000,1.0000,1.00,0.865000
3349,T001,book1.pdf,478,448,MIXED_COMPLEX,0.883519,0.750000,1.000000,1.0000,1.00,0.865000


## 4. Advanced Visual-Content Census

This section verifies that the corpus contains recoverable visual information.

It distinguishes:

- raster image occurrences;
- vector-heavy pages (important because many academic diagrams are vector graphics, not embedded JPEG/PNG files);
- repeated small assets that are likely logos/decorative elements;
- figure-caption pages;
- visual-dominant and mixed-complex pages.

No visual embedding model is selected here.

In [18]:
if visual_assets_df.empty:
    visual_asset_summary = pd.DataFrame(columns=["digest", "occurrences", "documents", "max_area_ratio", "median_area_ratio", "likely_repeated_decorative"])
else:
    visual_assets_df["asset_key"] = visual_assets_df.apply(lambda r: r["digest"] if str(r["digest"]).strip() else f"xref:{r['inventory_id']}:{r['xref']}:{r['width_px']}x{r['height_px']}", axis=1)
    visual_asset_summary = visual_assets_df.groupby("asset_key").agg(
        occurrences=("asset_key", "size"), documents=("inventory_id", "nunique"), max_area_ratio=("area_ratio", "max"),
        median_area_ratio=("area_ratio", "median"), max_width=("width_px", "max"), max_height=("height_px", "max")
    ).reset_index()
    visual_asset_summary["likely_repeated_decorative"] = (visual_asset_summary["occurrences"] >= 5) & (visual_asset_summary["max_area_ratio"] < 0.05)

visual_candidate_pages = pages_df[(pages_df["visual_score"] >= 0.30) | (pages_df["figure_caption_count"] > 0) | (pages_df["vector_drawing_count"] >= 20)].copy()
visual_candidate_pages = visual_candidate_pages.sort_values(["visual_score", "difficulty_score"], ascending=False)

visual_summary = pd.DataFrame([
    {"metric": "Raster image occurrences", "value": len(visual_assets_df)},
    {"metric": "Unique raster assets", "value": len(visual_asset_summary)},
    {"metric": "Likely repeated decorative assets", "value": int(visual_asset_summary["likely_repeated_decorative"].sum()) if len(visual_asset_summary) else 0},
    {"metric": "Pages with raster images", "value": int((pages_df["raster_image_count"] > 0).sum())},
    {"metric": "Pages with vector drawings", "value": int((pages_df["vector_drawing_count"] > 0).sum())},
    {"metric": "Figure-caption pages", "value": int((pages_df["figure_caption_count"] > 0).sum())},
    {"metric": "Visual candidate pages", "value": len(visual_candidate_pages)},
    {"metric": "Visual-dominant pages", "value": int((pages_df["page_modality"] == "VISUAL_DOMINANT").sum())}
])

display(visual_summary)
display(visual_candidate_pages.head(30)[["inventory_id", "filename", "pdf_page_number", "printed_page_label", "raster_image_count", "raster_area_ratio", "vector_drawing_count", "figure_caption_count", "visual_score", "page_modality"]])

assert len(visual_candidate_pages) > 0, "No visual candidates were detected; visual extraction readiness cannot be assessed."

,metric,value
0,Raster image occurrences,9768
1,Unique raster assets,4960
2,Likely repeated decorative assets,115
3,Pages with raster images,1977
4,Pages with vector drawings,5007
5,Figure-caption pages,2234
6,Visual candidate pages,2795
7,Visual-dominant pages,22


,inventory_id,filename,pdf_page_number,printed_page_label,raster_image_count,raster_area_ratio,vector_drawing_count,figure_caption_count,visual_score,page_modality
2983,T001,book1.pdf,112,82,3,0.1249,1115,3,1.0,MIXED_COMPLEX
3505,T001,book1.pdf,634,604,2,0.0896,10057,2,1.0,MIXED_COMPLEX
5847,T002,book2.pdf,965,931,2,0.0887,10932,2,1.0,MIXED_COMPLEX
241,L008,Landmark_16.pdf,7,,2,0.3061,3030,3,1.0,MIXED_COMPLEX
5876,T002,book2.pdf,994,960,2,0.1569,363,2,1.0,MIXED_COMPLEX
5171,T002,book2.pdf,289,255,3,0.1869,2230,2,1.0,MIXED_COMPLEX
3588,T001,book1.pdf,717,687,6,0.1235,93,3,1.0,MIXED_COMPLEX
3605,T001,book1.pdf,734,704,50,0.0976,1078,2,1.0,MIXED_COMPLEX
3407,T001,book1.pdf,536,506,3,0.4473,70,2,1.0,MIXED_COMPLEX
3359,T001,book1.pdf,488,458,1,0.0938,75,2,1.0,MIXED_COMPLEX


## 5. Mathematical-Content Census

This is a **math presence and difficulty audit**, not Math-RAG research.

Signals include:

- Unicode mathematical symbols;
- equation-like text lines;
- math-oriented font spans;
- math-rich and mixed-complex pages.

The output identifies the pages that the next notebook must use for equation/parser/Math-RAG experiments.

In [7]:
math_candidate_pages = pages_df[(pages_df["math_score"] >= 0.28) | (pages_df["equation_like_lines"] >= 2) | (pages_df["math_symbol_count"] >= 3) | (pages_df["math_font_spans"] >= 3)].copy()
math_candidate_pages = math_candidate_pages.sort_values(["math_score", "difficulty_score"], ascending=False)

math_doc_coverage = math_candidate_pages.groupby("inventory_id").size().rename("math_candidate_pages").reset_index()
math_summary = pd.DataFrame([
    {"metric": "Math candidate pages", "value": len(math_candidate_pages)},
    {"metric": "Documents with math candidates", "value": math_candidate_pages["inventory_id"].nunique()},
    {"metric": "Math-rich pages", "value": int((pages_df["page_modality"] == "MATH_RICH").sum())},
    {"metric": "Pages with equation-like lines", "value": int((pages_df["equation_like_lines"] > 0).sum())},
    {"metric": "Pages with math-font signals", "value": int((pages_df["math_font_spans"] > 0).sum())},
    {"metric": "Pages with mathematical Unicode", "value": int((pages_df["math_symbol_count"] > 0).sum())}
])

display(math_summary)
display(math_candidate_pages.head(30)[["inventory_id", "filename", "pdf_page_number", "printed_page_label", "math_symbol_count", "math_font_spans", "equation_like_lines", "math_score", "page_modality"]])

assert len(math_candidate_pages) > 0, "No mathematical-content candidates were detected."

,metric,value
0,Math candidate pages,4901
1,Documents with math candidates,53
2,Math-rich pages,2421
3,Pages with equation-like lines,4387
4,Pages with math-font signals,4276
5,Pages with mathematical Unicode,3826


,inventory_id,filename,pdf_page_number,printed_page_label,math_symbol_count,math_font_spans,equation_like_lines,math_score,page_modality
541,S007,survey_16.pdf,24,,12,12,12,1.0,MIXED_COMPLEX
97,L002,Landmark_10.pdf,5,,87,97,87,1.0,MIXED_COMPLEX
218,L022,Landmark_7.pdf,6,,27,238,27,1.0,MIXED_COMPLEX
464,S010,survey_19.pdf,8,,9,112,8,1.0,MIXED_COMPLEX
1272,S001,survey_1.pdf,23,,7,127,7,1.0,MIXED_COMPLEX
3334,T001,book1.pdf,463,433,9,76,11,1.0,MIXED_COMPLEX
98,L002,Landmark_10.pdf,6,,6,13,3,1.0,MIXED_COMPLEX
465,S010,survey_19.pdf,9,,8,651,7,1.0,MIXED_COMPLEX
3349,T001,book1.pdf,478,448,18,101,15,1.0,MIXED_COMPLEX
5617,T002,book2.pdf,735,701,9,106,8,1.0,MIXED_COMPLEX


## 6. Table & Numerical-Structure Readiness

A fast heuristic first identifies likely table pages.  
Where the installed PyMuPDF version supports `Page.find_tables()`, only the strongest candidate pages are validated with the built-in table detector to keep this audit practical.

This does **not** certify final table reconstruction fidelity.

In [8]:
table_candidate_pages = pages_df[(pages_df["table_score"] >= 0.28) | (pages_df["table_caption_count"] > 0)].copy()
table_candidate_pages = table_candidate_pages.sort_values(["table_score", "difficulty_score"], ascending=False)

table_validation_rows = []
table_engine_available = hasattr(fitz.Page, "find_tables")

if table_engine_available and len(table_candidate_pages):
    validate = table_candidate_pages.head(TABLE_VALIDATION_LIMIT)
    path_map = dict(zip(manifest["inventory_id"], manifest["resolved_path"]))
    for _, r in tqdm(validate.iterrows(), total=len(validate), desc="Validating table candidates"):
        detected = 0
        error = ""
        try:
            with fitz.open(path_map[r["inventory_id"]]) as doc:
                page = doc.load_page(int(r["page_index"]))
                finder = page.find_tables()
                detected = len(getattr(finder, "tables", []) or [])
        except Exception as e:
            error = str(e)
        table_validation_rows.append({"inventory_id": r["inventory_id"], "filename": r["filename"], "page_index": r["page_index"], "pdf_page_number": r["pdf_page_number"], "heuristic_table_score": r["table_score"], "detected_tables": detected, "error": error})

table_validation_df = pd.DataFrame(table_validation_rows, columns=["inventory_id", "filename", "page_index", "pdf_page_number", "heuristic_table_score", "detected_tables", "error"])

table_summary = pd.DataFrame([
    {"metric": "Table candidate pages", "value": len(table_candidate_pages)},
    {"metric": "Explicit Table-caption pages", "value": int((pages_df["table_caption_count"] > 0).sum())},
    {"metric": "Documents with table candidates", "value": table_candidate_pages["inventory_id"].nunique()},
    {"metric": "PyMuPDF table engine available", "value": table_engine_available},
    {"metric": "Candidate pages table-validated", "value": len(table_validation_df)},
    {"metric": "Pages with detected tables", "value": int((table_validation_df["detected_tables"] > 0).sum()) if len(table_validation_df) else 0}
])

display(table_summary)
display(table_candidate_pages.head(30)[["inventory_id", "filename", "pdf_page_number", "printed_page_label", "table_caption_count", "numeric_token_ratio", "vector_drawing_count", "table_score", "page_modality"]])

if len(table_validation_df):
    display(table_validation_df[table_validation_df["detected_tables"] > 0].head(30))

Validating table candidates:   0%|          | 0/600 [00:00<?, ?it/s]

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


,metric,value
0,Table candidate pages,2326
1,Explicit Table-caption pages,322
2,Documents with table candidates,51
3,PyMuPDF table engine available,True
4,Candidate pages table-validated,600
5,Pages with detected tables,237


,inventory_id,filename,pdf_page_number,printed_page_label,table_caption_count,numeric_token_ratio,vector_drawing_count,table_score,page_modality
541,S007,survey_16.pdf,24,,4,0.2845,104,1.0,MIXED_COMPLEX
97,L002,Landmark_10.pdf,5,,2,0.1344,270,1.0,MIXED_COMPLEX
218,L022,Landmark_7.pdf,6,,1,0.1404,63,1.0,MIXED_COMPLEX
230,L022,Landmark_7.pdf,18,,1,0.2623,835,1.0,MIXED_COMPLEX
1272,S001,survey_1.pdf,23,,2,0.1041,123,1.0,MIXED_COMPLEX
98,L002,Landmark_10.pdf,6,,4,0.0833,83,1.0,MIXED_COMPLEX
465,S010,survey_19.pdf,9,,1,0.1325,236,1.0,MIXED_COMPLEX
3349,T001,book1.pdf,478,448,1,0.1266,80,1.0,MIXED_COMPLEX
542,S007,survey_16.pdf,25,,2,0.1605,87,1.0,MIXED_COMPLEX
1982,T004,ESLII_print12_toc.pdf,292,,1,0.1493,60,1.0,MIXED_COMPLEX


,inventory_id,filename,page_index,pdf_page_number,heuristic_table_score,detected_tables,error
0,S007,survey_16.pdf,23,24,1.0,1,
1,L002,Landmark_10.pdf,4,5,1.0,5,
2,L022,Landmark_7.pdf,5,6,1.0,1,
3,L022,Landmark_7.pdf,17,18,1.0,1,
5,L002,Landmark_10.pdf,5,6,1.0,4,
6,S010,survey_19.pdf,8,9,1.0,1,
7,T001,book1.pdf,477,478,1.0,2,
12,L020,Landmark_5.pdf,4,5,1.0,2,
13,T001,book1.pdf,590,591,1.0,1,
14,L021,Landmark_6.pdf,7,8,1.0,4,


## 7. Layout, Multi-Column & Structural-Noise Audit

The corpus is profiled for layouts that are likely to challenge later parsers and retrieval pipelines:

- likely two-column pages;
- high block-count pages;
- reference/bibliography sections;
- TOC/index pages;
- pages with many vector objects;
- image-only or nearly blank pages.

These pages become explicit stress cases in the challenge set.

In [9]:
pages_df["image_only_or_scan_like"] = (pages_df["char_count"] < 120) & (pages_df["raster_area_ratio"] >= 0.70)
pages_df["minimal_page"] = (pages_df["char_count"] < 30) & (pages_df["visual_score"] < 0.20)
pages_df["reference_or_navigation"] = pages_df["reference_heading"] | pages_df["toc_heading"] | pages_df["index_heading"]

layout_candidates = pages_df[(pages_df["two_column_hint"]) | (pages_df["layout_score"] >= 0.45) | (pages_df["image_only_or_scan_like"])].copy()
layout_candidates = layout_candidates.sort_values(["layout_score", "difficulty_score"], ascending=False)

layout_summary = pd.DataFrame([
    {"metric": "Likely two-column pages", "value": int(pages_df["two_column_hint"].sum())},
    {"metric": "Image-only / scan-like pages", "value": int(pages_df["image_only_or_scan_like"].sum())},
    {"metric": "Reference-heading pages", "value": int(pages_df["reference_heading"].sum())},
    {"metric": "TOC-heading pages", "value": int(pages_df["toc_heading"].sum())},
    {"metric": "Index-heading pages", "value": int(pages_df["index_heading"].sum())},
    {"metric": "Minimal / nearly blank pages", "value": int(pages_df["minimal_page"].sum())},
    {"metric": "High-layout-complexity candidates", "value": len(layout_candidates)}
])

display(layout_summary)
display(layout_candidates.head(30)[["inventory_id", "filename", "pdf_page_number", "printed_page_label", "two_column_hint", "block_count", "raster_area_ratio", "vector_drawing_count", "layout_score", "difficulty_score"]])

,metric,value
0,Likely two-column pages,5718
1,Image-only / scan-like pages,2
2,Reference-heading pages,304
3,TOC-heading pages,65
4,Index-heading pages,74
5,Minimal / nearly blank pages,77
6,High-layout-complexity candidates,6019


,inventory_id,filename,pdf_page_number,printed_page_label,two_column_hint,block_count,raster_area_ratio,vector_drawing_count,layout_score,difficulty_score
541,S007,survey_16.pdf,24,,True,62,0.0216,104,1.0,0.903465
97,L002,Landmark_10.pdf,5,,True,130,0.0000,270,1.0,0.898000
218,L022,Landmark_7.pdf,6,,True,29,0.0000,63,1.0,0.892500
464,S010,survey_19.pdf,8,,True,115,0.2007,131,1.0,0.891225
230,L022,Landmark_7.pdf,18,,True,89,0.0678,835,1.0,0.881820
1272,S001,survey_1.pdf,23,,True,130,0.0433,123,1.0,0.875955
98,L002,Landmark_10.pdf,6,,True,37,0.0000,83,1.0,0.865000
465,S010,survey_19.pdf,9,,True,32,0.0000,236,1.0,0.865000
3349,T001,book1.pdf,478,448,True,31,0.0000,80,1.0,0.865000
5617,T002,book2.pdf,735,701,True,25,0.0000,127,1.0,0.858304


## 8. Unicode, Extraction Health & Page-Level Anomalies

This section looks for baseline extraction failures that could invalidate later RAG experiments.

It flags:

- high replacement-character counts;
- unexpectedly empty pages;
- very low text coverage by document;
- scan-like pages;
- page-level anomalies requiring special parser attention.

These are **review signals**, not automatic evidence that the source PDF is defective.

In [10]:
pages_df["replacement_ratio"] = pages_df["replacement_chars"]/pages_df["char_count"].replace(0, np.nan)
pages_df["anomaly_score"] = (
    (pages_df["replacement_ratio"].fillna(0) > 0.002).astype(int)*3 +
    (pages_df["image_only_or_scan_like"]).astype(int)*2 +
    (pages_df["minimal_page"]).astype(int) +
    ((pages_df["char_count"] == 0) & (pages_df["visual_score"] < 0.2)).astype(int)*2
)

problem_pages_df = pages_df[pages_df["anomaly_score"] > 0].copy().sort_values(["anomaly_score", "difficulty_score"], ascending=False)

doc_health = pages_df.groupby("inventory_id").agg(
    total_pages=("page_index", "size"),
    zero_text_pages=("char_count", lambda s: int((s == 0).sum())),
    minimal_pages=("minimal_page", "sum"),
    scan_like_pages=("image_only_or_scan_like", "sum"),
    replacement_chars=("replacement_chars", "sum"),
    two_column_pages=("two_column_hint", "sum")
).reset_index()

doc_health["zero_text_ratio"] = (doc_health["zero_text_pages"]/doc_health["total_pages"]).round(4)
doc_health["scan_like_ratio"] = (doc_health["scan_like_pages"]/doc_health["total_pages"]).round(4)
doc_health = doc_health.merge(manifest[["inventory_id", "filename", "category", "canonical_title"]], on="inventory_id", how="left")

health_summary = pd.DataFrame([
    {"metric": "Pages with replacement characters", "value": int((pages_df["replacement_chars"] > 0).sum())},
    {"metric": "Total replacement characters", "value": int(pages_df["replacement_chars"].sum())},
    {"metric": "Zero-text pages", "value": int((pages_df["char_count"] == 0).sum())},
    {"metric": "Problem/review pages", "value": len(problem_pages_df)},
    {"metric": "Documents above 10% zero-text pages", "value": int((doc_health["zero_text_ratio"] > 0.10).sum())}
])

display(health_summary)
display(doc_health.sort_values(["zero_text_ratio", "scan_like_ratio"], ascending=False).head(30))
if len(problem_pages_df):
    display(problem_pages_df.head(40)[["inventory_id", "filename", "pdf_page_number", "printed_page_label", "char_count", "raster_area_ratio", "replacement_chars", "image_only_or_scan_like", "anomaly_score"]])

,metric,value
0,Pages with replacement characters,2
1,Total replacement characters,11
2,Zero-text pages,65
3,Problem/review pages,80
4,Documents above 10% zero-text pages,0


,inventory_id,total_pages,zero_text_pages,minimal_pages,scan_like_pages,replacement_chars,two_column_pages,zero_text_ratio,scan_like_ratio,filename,category,canonical_title
29,S006,26,1,0,1,0,25,0.0385,0.0385,survey_15.pdf,Survey,Graph Neural Networks: A Review of Methods and Applications
47,T001,860,22,26,0,0,808,0.0256,0.0000,book1.pdf,Textbooks,Probabilistic Machine Learning: An Introduction
48,T002,1370,28,34,0,11,1289,0.0204,0.0000,book2.pdf,Textbooks,Probabilistic Machine Learning: Advanced Topics
51,T005,417,6,6,1,0,247,0.0144,0.0024,mml-book.pdf,Textbooks,Mathematics for Machine Learning
39,S016,160,2,2,0,0,136,0.0125,0.0000,survey_23.pdf,Survey,"Geometric Deep Learning: Grids, Groups, Graphs, Geodesics, and Gauges"
52,T006,541,4,4,0,0,397,0.0074,0.0000,UnderstandingDeepLearning_02_09_26_C.pdf,Textbooks,Understanding Deep Learning
49,T003,1151,2,2,0,0,706,0.0017,0.0000,d2l-en.pdf,Textbooks,Dive into Deep Learning
0,L001,15,0,0,0,0,14,0.0000,0.0000,Landmark_1.pdf,Landmark,Attention Is All You Need
1,L002,12,0,0,0,0,12,0.0000,0.0000,Landmark_10.pdf,Landmark,Deep Residual Learning for Image Recognition
2,L003,10,0,0,0,0,9,0.0000,0.0000,Landmark_11.pdf,Landmark,"You Only Look Once: Unified, Real-Time Object Detection"


,inventory_id,filename,pdf_page_number,printed_page_label,char_count,raster_area_ratio,replacement_chars,image_only_or_scan_like,anomaly_score
5137,T002,book2.pdf,255,221,2305,0.0,8,False,3
6254,T006,UnderstandingDeepLearning_02_09_26_C.pdf,2,ii,0,0.0,0,False,3
6256,T006,UnderstandingDeepLearning_02_09_26_C.pdf,4,ii,0,0.0,0,False,3
6262,T006,UnderstandingDeepLearning_02_09_26_C.pdf,10,viii,0,0.0,0,False,3
6264,T006,UnderstandingDeepLearning_02_09_26_C.pdf,12,x,0,0.0,0,False,3
788,S016,survey_23.pdf,2,ii,0,0.0,0,False,3
918,S016,survey_23.pdf,132,128,0,0.0,0,False,3
2456,T005,mml-book.pdf,2,2,0,0.0,0,False,3
2460,T005,mml-book.pdf,6,iv,0,0.0,0,False,3
2468,T005,mml-book.pdf,14,8,0,0.0,0,False,3


## 9. Source-Locator & Provenance Readiness

Future grounded RAG answers must be able to point back to the exact source.

This audit verifies that every page can be represented by a stable locator containing:

- canonical document ID;
- PDF page index;
- human-facing PDF page number;
- printed page label when available;
- canonical work ID;
- source filename.

Bounding boxes for final extracted objects will be researched/validated later.

In [11]:
locator_columns = ["inventory_id", "canonical_work_id", "filename", "page_index", "pdf_page_number", "printed_page_label"]
locator_df = pages_df[locator_columns].copy()
locator_df["locator_id"] = locator_df.apply(lambda r: f"{r['inventory_id']}:p{int(r['page_index']):05d}", axis=1)

locator_checks = {
    "Locator IDs unique": locator_df["locator_id"].is_unique,
    "Every page has document ID": locator_df["inventory_id"].fillna("").ne("").all(),
    "Every page has canonical work ID": locator_df["canonical_work_id"].fillna("").ne("").all(),
    "Every page has physical page number": locator_df["pdf_page_number"].notna().all(),
    "Every document represented": locator_df["inventory_id"].nunique() == TARGET_TOTAL,
    "Locator rows equal physical pages": len(locator_df) == int(manifest["page_count"].sum())
}

locator_checks_df = pd.DataFrame([{"check": k, "status": "PASS" if v else "FAIL"} for k, v in locator_checks.items()])
display(locator_checks_df)
assert all(locator_checks.values()), "Source-locator readiness failed."

,check,status
0,Locator IDs unique,PASS
1,Every page has document ID,PASS
2,Every page has canonical work ID,PASS
3,Every page has physical page number,PASS
4,Every document represented,PASS
5,Locator rows equal physical pages,PASS


## 10. Document-Level Content & Modality Profile

The page census is aggregated into a single document-level profile.

This shows which documents are text-heavy, math-heavy, visual-heavy, table-rich, or layout-complex and ensures the next research notebook can sample all important content types.

In [12]:
document_profile = pages_df.groupby("inventory_id").agg(
    pages=("page_index", "size"),
    text_rich_pages=("page_modality", lambda s: int((s == "TEXT_RICH").sum())),
    visual_rich_pages=("page_modality", lambda s: int(s.isin(["VISUAL_RICH", "VISUAL_DOMINANT"]).sum())),
    math_rich_pages=("page_modality", lambda s: int((s == "MATH_RICH").sum())),
    table_rich_pages=("page_modality", lambda s: int((s == "TABLE_RICH").sum())),
    mixed_complex_pages=("page_modality", lambda s: int((s == "MIXED_COMPLEX").sum())),
    raster_occurrences=("raster_image_count", "sum"),
    vector_drawings=("vector_drawing_count", "sum"),
    figure_caption_pages=("figure_caption_count", lambda s: int((s > 0).sum())),
    table_caption_pages=("table_caption_count", lambda s: int((s > 0).sum())),
    math_signal_pages=("math_score", lambda s: int((s >= 0.28).sum())),
    table_candidate_pages=("table_score", lambda s: int((s >= 0.28).sum())),
    two_column_pages=("two_column_hint", "sum"),
    scan_like_pages=("image_only_or_scan_like", "sum"),
    max_difficulty=("difficulty_score", "max"),
    mean_difficulty=("difficulty_score", "mean")
).reset_index()

document_profile = document_profile.merge(manifest[["inventory_id", "filename", "canonical_title", "category", "source_type", "primary_topic", "publication_year"]], on="inventory_id", how="left")
document_profile["mean_difficulty"] = document_profile["mean_difficulty"].round(4)
document_profile["max_difficulty"] = document_profile["max_difficulty"].round(4)

display(document_profile.sort_values("max_difficulty", ascending=False))

,inventory_id,pages,text_rich_pages,visual_rich_pages,math_rich_pages,table_rich_pages,mixed_complex_pages,raster_occurrences,vector_drawings,figure_caption_pages,table_caption_pages,math_signal_pages,table_candidate_pages,two_column_pages,scan_like_pages,max_difficulty,mean_difficulty,filename,canonical_title,category,source_type,primary_topic,publication_year
30,S007,31,11,1,11,0,8,1038,866,7,4,21,7,30,0,0.9035,0.4386,survey_16.pdf,A Comprehensive Survey on Transfer Learning,Survey,survey,General_ML,2019
1,L002,12,2,1,1,0,8,0,1364,7,7,10,9,12,0,0.8980,0.6459,Landmark_10.pdf,Deep Residual Learning for Image Recognition,Landmark,landmark,Computer_Vision,2015
21,L022,22,4,0,2,2,14,300,2813,12,10,15,15,21,0,0.8925,0.5366,Landmark_7.pdf,An Image Is Worth 16x16 Words: Transformers for Image Recognition at Scale,Landmark,landmark,Computer_Vision,2020
33,S010,35,13,5,0,1,16,321,7131,17,12,21,18,35,0,0.8912,0.5160,survey_19.pdf,Deep Learning for Generic Object Detection: A Survey,Survey,survey,Computer_Vision,2018
24,S001,144,102,9,14,4,15,236,3011,17,20,47,25,142,0,0.8760,0.3012,survey_1.pdf,A Survey of Large Language Models,Survey,survey,NLP_LLM,2023
47,T001,860,96,14,417,0,305,1097,492943,400,28,742,334,808,0,0.8676,0.5252,book1.pdf,Probabilistic Machine Learning: An Introduction,Textbooks,textbook,General_ML,2022
48,T002,1370,209,24,717,0,385,1579,245017,534,27,1159,415,1289,0,0.8583,0.4939,book2.pdf,Probabilistic Machine Learning: Advanced Topics,Textbooks,textbook,General_ML,2023
50,T004,764,134,32,377,6,204,180,397440,341,41,759,272,673,0,0.8540,0.4754,ESLII_print12_toc.pdf,"The Elements of Statistical Learning: Data Mining, Inference, and Prediction",Textbooks,textbook,General_ML,2009
7,L008,25,4,9,6,0,6,22,3212,16,3,14,14,16,0,0.8475,0.4402,Landmark_16.pdf,Denoising Diffusion Probabilistic Models,Landmark,landmark,Generative_AI,2020
19,L020,13,4,0,2,3,4,0,168,1,7,7,7,13,0,0.8390,0.3921,Landmark_5.pdf,Dense Passage Retrieval for Open-Domain Question Answering,Landmark,landmark,RAG_IR,2020


## 11. Corpus-Level Topic × Modality Coverage

This verifies that content complexity is not concentrated in only one topic or source type.

The result is descriptive. A topic does not fail merely because it contains fewer figures or equations.

In [13]:
topic_modality = pages_df.groupby("primary_topic").agg(
    documents=("inventory_id", "nunique"),
    pages=("page_index", "size"),
    visual_candidates=("visual_score", lambda s: int((s >= 0.30).sum())),
    math_candidates=("math_score", lambda s: int((s >= 0.28).sum())),
    table_candidates=("table_score", lambda s: int((s >= 0.28).sum())),
    two_column_pages=("two_column_hint", "sum"),
    mixed_complex_pages=("page_modality", lambda s: int((s == "MIXED_COMPLEX").sum()))
).reset_index()

category_modality = pages_df.groupby("category").agg(
    documents=("inventory_id", "nunique"),
    pages=("page_index", "size"),
    visual_candidates=("visual_score", lambda s: int((s >= 0.30).sum())),
    math_candidates=("math_score", lambda s: int((s >= 0.28).sum())),
    table_candidates=("table_score", lambda s: int((s >= 0.28).sum())),
    two_column_pages=("two_column_hint", "sum"),
    scan_like_pages=("image_only_or_scan_like", "sum")
).reset_index()

display(topic_modality)
display(category_modality)

,primary_topic,documents,pages,visual_candidates,math_candidates,table_candidates,two_column_pages,mixed_complex_pages
0,Computer_Vision,9,188,94,103,95,177,78
1,Deep_Learning,7,1807,831,1103,828,1213,409
2,GNN,3,200,33,118,22,173,16
3,General_ML,7,3551,1245,3101,1192,3153,1009
4,Generative_AI,4,107,40,51,25,91,12
5,NLP_LLM,15,818,162,206,133,796,86
6,RAG_IR,3,53,12,15,12,51,8
7,Reinforcement_Learning,5,70,17,45,19,64,17


,category,documents,pages,visual_candidates,math_candidates,table_candidates,two_column_pages,scan_like_pages
0,Landmark,24,441,174,268,200,392,0
1,Survey,23,1250,249,419,165,1206,1
2,Textbooks,6,5103,2011,4055,1961,4120,1


## 12. Advanced Challenge / Sentinel Page Set

The next notebook should not randomly test parsers or RAG modalities.

This cell creates a reproducible challenge set containing, per document, the strongest available examples of:

- normal text;
- visual content;
- mathematical content;
- tables/numerical structure;
- layout complexity;
- overall difficult/mixed content.

Textbooks receive a larger page budget than papers/surveys.  
A small set of globally difficult pages is then added.

The final output is the **research benchmark page manifest** for the next notebook.

In [14]:
def select_for_document(g, budget):
    chosen, reasons = {}, {}

    def choose(score_col, reason, mask=None):
        pool = g if mask is None else g[mask]
        if pool.empty: return
        idx = pool[score_col].idxmax()
        key = int(pool.loc[idx, "page_index"])
        if key not in chosen:
            chosen[key] = idx
            reasons[key] = [reason]
        else:
            reasons[key].append(reason)

    choose("text_score", "TEXT_REPRESENTATIVE", g["char_count"] >= g["char_count"].median())
    choose("visual_score", "VISUAL_STRESS", (g["visual_score"] >= 0.20) | (g["figure_caption_count"] > 0))
    choose("math_score", "MATH_STRESS", (g["math_score"] >= 0.15) | (g["equation_like_lines"] > 0))
    choose("table_score", "TABLE_NUMERIC_STRESS", (g["table_score"] >= 0.15) | (g["table_caption_count"] > 0))
    choose("layout_score", "LAYOUT_STRESS")
    choose("difficulty_score", "OVERALL_DIFFICULT")

    ranked = g.sort_values("difficulty_score", ascending=False)
    for idx, r in ranked.iterrows():
        if len(chosen) >= budget: break
        key = int(r["page_index"])
        if key not in chosen:
            chosen[key] = idx
            reasons[key] = ["DIFFICULTY_FILL"]

    rows = []
    for page_index, idx in chosen.items():
        r = g.loc[idx].copy()
        r["challenge_reason"] = " + ".join(reasons[page_index])
        rows.append(r)

    return pd.DataFrame(rows)

challenge_parts = []
for doc_id, g in pages_df.groupby("inventory_id", sort=False):
    category = g["category"].iloc[0]
    budget = TEXTBOOK_CHALLENGE_PAGES if category == "Textbooks" else PAPER_CHALLENGE_PAGES
    challenge_parts.append(select_for_document(g, budget))

challenge_pages_df = pd.concat(challenge_parts, ignore_index=True)

existing = set(zip(challenge_pages_df["inventory_id"], challenge_pages_df["page_index"]))
global_extra = pages_df.sort_values("difficulty_score", ascending=False)
extra_rows = []
for _, r in global_extra.iterrows():
    key = (r["inventory_id"], r["page_index"])
    if key in existing: continue
    rr = r.copy(); rr["challenge_reason"] = "GLOBAL_TOP_DIFFICULTY"
    extra_rows.append(rr); existing.add(key)
    if len(extra_rows) >= GLOBAL_EXTRA_CHALLENGE_PAGES: break

if extra_rows:
    challenge_pages_df = pd.concat([challenge_pages_df, pd.DataFrame(extra_rows)], ignore_index=True)

challenge_pages_df["challenge_id"] = challenge_pages_df.apply(lambda r: f"{r['inventory_id']}_p{int(r['page_index']):05d}", axis=1)
challenge_pages_df = challenge_pages_df.sort_values(["inventory_id", "page_index"]).reset_index(drop=True)

challenge_columns = [
    "challenge_id", "inventory_id", "canonical_work_id", "filename", "canonical_title", "category", "primary_topic",
    "page_index", "pdf_page_number", "printed_page_label", "challenge_reason", "page_modality", "text_score", "visual_score",
    "math_score", "table_score", "layout_score", "difficulty_score", "raster_image_count", "raster_area_ratio",
    "vector_drawing_count", "equation_like_lines", "table_caption_count", "figure_caption_count", "two_column_hint"
]
challenge_pages_df = challenge_pages_df[challenge_columns]

challenge_coverage = challenge_pages_df.groupby("inventory_id").size()
print("Challenge pages          :", len(challenge_pages_df))
print("Documents represented    :", challenge_pages_df["inventory_id"].nunique(), "/", TARGET_TOTAL)
print("Min pages per document   :", int(challenge_coverage.min()))
print("Max pages per document   :", int(challenge_coverage.max()))

display(challenge_pages_df.head(40))
assert challenge_pages_df["inventory_id"].nunique() == TARGET_TOTAL, "Challenge set does not cover every document."

Challenge pages          : 276
Documents represented    : 53 / 53
Min pages per document   : 4
Max pages per document   : 15


,challenge_id,inventory_id,canonical_work_id,filename,canonical_title,category,primary_topic,page_index,pdf_page_number,printed_page_label,challenge_reason,page_modality,text_score,visual_score,math_score,table_score,layout_score,difficulty_score,raster_image_count,raster_area_ratio,vector_drawing_count,equation_like_lines,table_caption_count,figure_caption_count,two_column_hint
0,L001_p00001,L001,attention_is_all_you_need_2017,Landmark_1.pdf,Attention Is All You Need,Landmark,NLP_LLM,1,2,,TEXT_REPRESENTATIVE,TEXT_RICH,0.981084,0.000000,0.350000,0.025600,0.680000,0.279008,0,0.0000,0,0,0,0,True
1,L001_p00004,L001,attention_is_all_you_need_2017,Landmark_1.pdf,Attention Is All You Need,Landmark,NLP_LLM,4,5,,MATH_STRESS,MATH_RICH,0.947345,0.008333,1.000000,0.045067,0.755556,0.461501,0,0.0000,1,7,0,0,True
2,L001_p00008,L001,attention_is_all_you_need_2017,Landmark_1.pdf,Attention Is All You Need,Landmark,NLP_LLM,8,9,,TABLE_NUMERIC_STRESS + OVERALL_DIFFICULT,MIXED_COMPLEX,0.938963,0.441667,0.530556,1.000000,0.980000,0.678900,0,0.0000,53,1,1,0,True
3,L001_p00012,L001,attention_is_all_you_need_2017,Landmark_1.pdf,Attention Is All You Need,Landmark,NLP_LLM,12,13,,VISUAL_STRESS + LAYOUT_STRESS,VISUAL_RICH,0.783954,0.750000,0.000000,0.378000,1.000000,0.513040,0,0.0000,624,0,0,1,True
4,L002_p00000,L002,deep_residual_learning_for_image_recognition_2015,Landmark_10.pdf,Deep Residual Learning for Image Recognition,Landmark,Computer_Vision,0,1,,LAYOUT_STRESS,VISUAL_RICH,0.981964,0.691667,0.347222,0.436800,1.000000,0.594124,0,0.0000,59,1,0,1,True
5,L002_p00001,L002,deep_residual_learning_for_image_recognition_2015,Landmark_10.pdf,Deep Residual Learning for Image Recognition,Landmark,Computer_Vision,1,2,,TEXT_REPRESENTATIVE,TEXT_RICH,1.000000,0.333333,0.350000,0.135067,0.838889,0.416534,0,0.0000,16,0,0,1,True
6,L002_p00003,L002,deep_residual_learning_for_image_recognition_2015,Landmark_10.pdf,Deep Residual Learning for Image Recognition,Landmark,Computer_Vision,3,4,,VISUAL_STRESS,MIXED_COMPLEX,0.989159,0.900000,0.891667,0.600000,1.000000,0.800000,0,0.0000,366,3,0,2,True
7,L002_p00004,L002,deep_residual_learning_for_image_recognition_2015,Landmark_10.pdf,Deep Residual Learning for Image Recognition,Landmark,Computer_Vision,4,5,,MATH_STRESS + TABLE_NUMERIC_STRESS + OVERALL_DIFFICULT,MIXED_COMPLEX,0.970557,0.900000,1.000000,1.000000,1.000000,0.898000,0,0.0000,270,87,2,2,True
8,L002_p00005,L002,deep_residual_learning_for_image_recognition_2015,Landmark_10.pdf,Deep Residual Learning for Image Recognition,Landmark,Computer_Vision,5,6,,GLOBAL_TOP_DIFFICULTY,MIXED_COMPLEX,0.989056,0.750000,1.000000,1.000000,1.000000,0.865000,0,0.0000,83,3,4,1,True
9,L003_p00000,L003,you_only_look_once_unified_real_time_object_detection_2015,Landmark_11.pdf,"You Only Look Once: Unified, Real-Time Object Detection",Landmark,Computer_Vision,0,1,,LAYOUT_STRESS,MIXED_COMPLEX,0.985284,0.766560,0.530556,0.319600,1.000000,0.633505,2,0.0144,68,1,0,1,True


## 13. Optional Challenge-Page Preview Export

Disabled by default because rendering 200+ pages creates many image files.

Set:

```python
EXPORT_CHALLENGE_PREVIEWS = True
```

in the configuration cell when visual manual review is needed.  
The exported page renders are intended only for research/audit, not as final figure crops.

In [15]:
preview_dir = OUT_DIR/"challenge_previews"

if EXPORT_CHALLENGE_PREVIEWS:
    preview_dir.mkdir(parents=True, exist_ok=True)
    path_map = dict(zip(manifest["inventory_id"], manifest["resolved_path"]))
    for _, r in tqdm(challenge_pages_df.iterrows(), total=len(challenge_pages_df), desc="Rendering challenge pages"):
        out_path = preview_dir/f"{r['challenge_id']}.png"
        if out_path.exists(): continue
        with fitz.open(path_map[r["inventory_id"]]) as doc:
            page = doc.load_page(int(r["page_index"]))
            pix = page.get_pixmap(dpi=PREVIEW_DPI, alpha=False)
            pix.save(out_path)
    print("Rendered:", len(list(preview_dir.glob("*.png"))), "challenge previews")
else:
    print("Preview export skipped. Set EXPORT_CHALLENGE_PREVIEWS=True when manual visual inspection is required.")

Preview export skipped. Set EXPORT_CHALLENGE_PREVIEWS=True when manual visual inspection is required.


## 14. Final Dataset-Readiness Risk Audit

This section converts the entire Notebook 02 audit into explicit **critical gates** and **non-blocking warnings**.

### Critical gates

A failure means the corpus is not ready for the next RAG research notebook.

### Warnings

Warnings describe difficult content that the next notebook must explicitly test.  
They do not imply that the source corpus is bad.

In [16]:
critical_checks = []
warning_checks = []

def critical(name, passed, detail=""):
    critical_checks.append({"check": name, "passed": bool(passed), "detail": str(detail)})

def warning(name, passed, detail=""):
    warning_checks.append({"check": name, "passed": bool(passed), "detail": str(detail)})

critical("Notebook 01 certification preserved", all(gate_01.values()), gate_01)
critical("All 53 documents represented", pages_df["inventory_id"].nunique() == TARGET_TOTAL, pages_df["inventory_id"].nunique())
critical("Every physical page profiled", len(pages_df) == int(manifest["page_count"].sum()), f"{len(pages_df)}/{int(manifest['page_count'].sum())}")
critical("Zero page-scan exceptions", scan_errors_df.empty, len(scan_errors_df))
critical("Source locators complete", all(locator_checks.values()), locator_checks)
critical("Every document has extractable text somewhere", pages_df.groupby("inventory_id")["char_count"].max().gt(500).all())
critical("No document exceeds 20% zero-text pages", (doc_health["zero_text_ratio"] <= 0.20).all(), doc_health.loc[doc_health["zero_text_ratio"] > 0.20, ["inventory_id", "zero_text_ratio"]].to_dict("records"))
critical("Visual-content candidates exist", len(visual_candidate_pages) > 0, len(visual_candidate_pages))
critical("Mathematical-content candidates exist", len(math_candidate_pages) > 0, len(math_candidate_pages))
critical("Table/numerical candidates exist", len(table_candidate_pages) > 0, len(table_candidate_pages))
critical("Challenge set covers all documents", challenge_pages_df["inventory_id"].nunique() == TARGET_TOTAL, challenge_pages_df["inventory_id"].nunique())
critical("Challenge set contains visual stress pages", challenge_pages_df["visual_score"].ge(0.20).any())
critical("Challenge set contains math stress pages", challenge_pages_df["math_score"].ge(0.15).any())
critical("Challenge set contains table/numeric stress pages", challenge_pages_df["table_score"].ge(0.15).any())
critical("Challenge set contains layout stress pages", challenge_pages_df["layout_score"].ge(0.25).any())

warning("No document exceeds 10% zero-text pages", (doc_health["zero_text_ratio"] <= 0.10).all(), doc_health.loc[doc_health["zero_text_ratio"] > 0.10, ["inventory_id", "zero_text_ratio"]].to_dict("records"))
warning("Unicode replacement corruption remains very low", pages_df["replacement_chars"].sum() <= 50, int(pages_df["replacement_chars"].sum()))
warning("PyMuPDF table detector available", table_engine_available, "Optional validation layer")
warning("No scan-like pages", int(pages_df["image_only_or_scan_like"].sum()) == 0, int(pages_df["image_only_or_scan_like"].sum()))
warning("No likely two-column pages", int(pages_df["two_column_hint"].sum()) == 0, int(pages_df["two_column_hint"].sum()))

critical_df = pd.DataFrame(critical_checks)
warning_df = pd.DataFrame(warning_checks)
critical_failures = critical_df[~critical_df["passed"]]
warning_failures = warning_df[~warning_df["passed"]]

display(critical_df)
display(warning_df)

print("Critical failures :", len(critical_failures))
print("Warnings triggered:", len(warning_failures))

,check,passed,detail
0,Notebook 01 certification preserved,True,"{'Notebook 01 certified': True, 'Notebook 01 ready': True, '53 manifest documents': True, 'Category counts exact': True, 'All physical f..."
1,All 53 documents represented,True,53
2,Every physical page profiled,True,6794/6794
3,Zero page-scan exceptions,True,0
4,Source locators complete,True,"{'Locator IDs unique': True, 'Every page has document ID': np.True_, 'Every page has canonical work ID': np.True_, 'Every page has physi..."
5,Every document has extractable text somewhere,True,
6,No document exceeds 20% zero-text pages,True,[]
7,Visual-content candidates exist,True,2795
8,Mathematical-content candidates exist,True,4901
9,Table/numerical candidates exist,True,2326


,check,passed,detail
0,No document exceeds 10% zero-text pages,True,[]
1,Unicode replacement corruption remains very low,True,11
2,PyMuPDF table detector available,True,Optional validation layer
3,No scan-like pages,False,2
4,No likely two-column pages,False,5718


Critical failures : 0
Warnings triggered: 2


## 15. Final Notebook 02 Certification & Export

A clean critical gate means:

> **The dataset-checking phase is complete, and the corpus is ready for dedicated Text-RAG, Math-RAG, Visual-RAG, parser, retrieval, and full architecture research.**

It does **not** mean that final parser fidelity or RAG accuracy has already been proven.

In [17]:
ready = critical_failures.empty
status = "READY FOR RAG RESEARCH ✅" if ready else "NOT READY ❌"

content_summary = {
    "project": "RAGScholar",
    "notebook": "02_Content_Readiness_and_Challenge_Set",
    "corpus_snapshot_sha256": snapshot_now,
    "documents": int(pages_df["inventory_id"].nunique()),
    "physical_pages": int(len(pages_df)),
    "raster_asset_occurrences": int(len(visual_assets_df)),
    "unique_raster_assets": int(len(visual_asset_summary)),
    "visual_candidate_pages": int(len(visual_candidate_pages)),
    "math_candidate_pages": int(len(math_candidate_pages)),
    "table_candidate_pages": int(len(table_candidate_pages)),
    "likely_two_column_pages": int(pages_df["two_column_hint"].sum()),
    "scan_like_pages": int(pages_df["image_only_or_scan_like"].sum()),
    "zero_text_pages": int((pages_df["char_count"] == 0).sum()),
    "replacement_characters": int(pages_df["replacement_chars"].sum()),
    "challenge_pages": int(len(challenge_pages_df)),
    "challenge_documents": int(challenge_pages_df["inventory_id"].nunique()),
    "critical_checks": int(len(critical_df)),
    "critical_failures": int(len(critical_failures)),
    "warnings_triggered": int(len(warning_failures)),
    "ready_for_rag_research": bool(ready),
    "status": status
}

pages_df.to_csv(OUT_DIR/"page_content_profile.csv", index=False, encoding="utf-8-sig")
visual_assets_df.to_csv(OUT_DIR/"visual_asset_inventory.csv", index=False, encoding="utf-8-sig")
visual_asset_summary.to_csv(OUT_DIR/"visual_asset_summary.csv", index=False, encoding="utf-8-sig")
document_profile.to_csv(OUT_DIR/"document_content_profile.csv", index=False, encoding="utf-8-sig")
visual_candidate_pages.to_csv(OUT_DIR/"visual_candidate_pages.csv", index=False, encoding="utf-8-sig")
math_candidate_pages.to_csv(OUT_DIR/"math_candidate_pages.csv", index=False, encoding="utf-8-sig")
table_candidate_pages.to_csv(OUT_DIR/"table_candidate_pages.csv", index=False, encoding="utf-8-sig")
table_validation_df.to_csv(OUT_DIR/"table_candidate_validation.csv", index=False, encoding="utf-8-sig")
layout_candidates.to_csv(OUT_DIR/"layout_challenge_pages.csv", index=False, encoding="utf-8-sig")
problem_pages_df.to_csv(OUT_DIR/"problem_pages_review.csv", index=False, encoding="utf-8-sig")
topic_modality.to_csv(OUT_DIR/"topic_modality_coverage.csv", index=False, encoding="utf-8-sig")
category_modality.to_csv(OUT_DIR/"category_modality_coverage.csv", index=False, encoding="utf-8-sig")
challenge_pages_df.to_csv(OUT_DIR/"challenge_pages.csv", index=False, encoding="utf-8-sig")
critical_df.to_csv(OUT_DIR/"readiness_critical_checks.csv", index=False, encoding="utf-8-sig")
warning_df.to_csv(OUT_DIR/"readiness_warning_checks.csv", index=False, encoding="utf-8-sig")

with open(OUT_DIR/"content_readiness_summary.json", "w", encoding="utf-8") as f:
    json.dump(content_summary, f, indent=2, ensure_ascii=False)

research_contract = {
    "certified_input": "metadata/corpus_manifest.jsonl",
    "page_profile": "metadata/notebook02/page_content_profile.csv",
    "challenge_set": "metadata/notebook02/challenge_pages.csv",
    "visual_candidates": "metadata/notebook02/visual_candidate_pages.csv",
    "math_candidates": "metadata/notebook02/math_candidate_pages.csv",
    "table_candidates": "metadata/notebook02/table_candidate_pages.csv",
    "layout_candidates": "metadata/notebook02/layout_challenge_pages.csv",
    "next_stage": [
        "parser fidelity research",
        "text retrieval research",
        "math retrieval research",
        "visual retrieval research",
        "table/numeric retrieval research",
        "RAGFlow integration",
        "full RAGScholar architecture implementation"
    ]
}

with open(OUT_DIR/"next_notebook_contract.json", "w", encoding="utf-8") as f:
    json.dump(research_contract, f, indent=2, ensure_ascii=False)

print("="*88)
print("RAGSCHOLAR — DATASET CONTENT READINESS CERTIFICATION")
print("="*88)
print(f"Documents                    : {content_summary['documents']} / {TARGET_TOTAL}")
print(f"Physical pages               : {content_summary['physical_pages']:,}")
print(f"Visual candidate pages       : {content_summary['visual_candidate_pages']:,}")
print(f"Math candidate pages         : {content_summary['math_candidate_pages']:,}")
print(f"Table candidate pages        : {content_summary['table_candidate_pages']:,}")
print(f"Likely two-column pages      : {content_summary['likely_two_column_pages']:,}")
print(f"Challenge benchmark pages    : {content_summary['challenge_pages']:,}")
print(f"Challenge document coverage  : {content_summary['challenge_documents']} / {TARGET_TOTAL}")
print("-"*88)
print(f"Critical checks              : {content_summary['critical_checks']}")
print(f"Critical failures            : {content_summary['critical_failures']}")
print(f"Non-blocking warnings        : {content_summary['warnings_triggered']}")
print("-"*88)
print("FINAL STATUS                 :", status)
print("="*88)

print("\nSaved under:", OUT_DIR)
for name in sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()):
    print("✓", name)

if ready:
    print("\nDataset-checking phase is complete. Next notebook may begin Text / Math / Visual / Parser / RAG architecture research.")

RAGSCHOLAR — DATASET CONTENT READINESS CERTIFICATION
Documents                    : 53 / 53
Physical pages               : 6,794
Visual candidate pages       : 2,795
Math candidate pages         : 4,901
Table candidate pages        : 2,326
Likely two-column pages      : 5,718
Challenge benchmark pages    : 276
Challenge document coverage  : 53 / 53
----------------------------------------------------------------------------------------
Critical checks              : 15
Critical failures            : 0
Non-blocking warnings        : 2
----------------------------------------------------------------------------------------
FINAL STATUS                 : READY FOR RAG RESEARCH ✅

Saved under: C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\metadata\notebook02
✓ category_modality_coverage.csv
✓ challenge_pages.csv
✓ content_readiness_summary.json
✓ document_content_profile.csv
✓ layout_challenge_pages.csv
✓ math_candidate_pages.csv
✓ next_notebook_contract.json
✓ page_con

# Notebook 02 Completion Meaning

If the final status is:

```text
READY FOR RAG RESEARCH ✅
```

then the project has two independently completed data-readiness layers:

1. **Notebook 01 — Physical Source Integrity**
   - correct files;
   - valid PDFs;
   - canonical identity;
   - duplicates;
   - metadata;
   - reproducible hashes.

2. **Notebook 02 — Content & Modality Readiness**
   - every page inventoried;
   - text presence profiled;
   - image/vector content profiled;
   - mathematical content profiled;
   - table/numerical candidates profiled;
   - difficult layouts identified;
   - exact page locators established;
   - challenge benchmark created.

The next notebook can therefore focus on **research and implementation**, not basic dataset uncertainty.